# 09 · Detectar o coração com YOLO

Curso **"YOLO na prática"** do professor (aula 1, detecção em imagem; aula 4, treino com dataset
próprio) e as aulas **"A caixa e o IoU"** e **"Confiança e a briga das caixas"** (NMS). Até aqui o
coração foi medido por larguras (04), apontado por mapas de calor (07) e recortado (08). Agora um
**detector** aprende a pôr uma caixa nele.

1. **o formato YOLO**: uma linha por objeto, medidas normalizadas, e o caminho de volta para pixels;
2. **um dataset próprio sem rotulador humano**: as caixas de treino são **pseudo-rótulos** tirados das
   bordas do ICT do notebook 04;
3. **treino** do YOLOv8n a partir dos pesos do COCO, e a melhor época;
4. **detecção em imagem**: caixas, confiança, o efeito do limiar e do NMS;
5. **avaliação contra o gabarito**: IoU e mAP50 nas caixas dos 29 contornos marcados à mão. O aluno
   (YOLO) consegue superar o professor (os pseudo-rótulos)?
6. **o ICT da caixa**: a largura da caixa sobre a largura do tórax separa as classes?

Vídeo e rastreamento (aulas 2 e 3 do curso) ficam de fora: raio-X de tórax é imagem parada.

**Sem espelhar:** como na CNN do notebook 05, o espelhamento horizontal do YOLO fica desligado
(`fliplr=0`), porque põe o coração do lado errado. O mosaico (quatro imagens coladas numa) também,
porque cria tórax impossíveis.

In [ ]:
import sys
from pathlib import Path

# Permite importar `src` mesmo fora do ambiente do uv (ex.: Colab com o repositório clonado).
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import logging
import urllib.request
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO
from ultralytics.utils import LOGGER

from src import config
from src.data.dados import baixar_base, carregar_imagens, dividir, listar_imagens
from src.model import avaliacao
from src.utils.io import configurar_estilo, salvar_figura, salvar_tabela
from src.visao import deteccao, ict, segmentacao

configurar_estilo()
LOGGER.setLevel(logging.WARNING)  # o YOLO escreve uma linha por lote; o que importa vai para results.csv
warnings.filterwarnings("ignore", message=".*deterministic.*")
SECAO = "09-deteccao-yolo"
PARAMS_ICT = dict(sigma_vertical=3, linhas=(50, 90), quantil=75)
EPOCAS = 40
# True refaz o treino; False reaproveita models/yolo/coracao/weights/best.pt se existir.
RETREINAR = False
DISPOSITIVO = "mps" if torch.backends.mps.is_available() else "cpu"

baixar_base()
catalogo = dividir(listar_imagens())
imagens = carregar_imagens(catalogo["caminho"])
y = catalogo["classe"].to_numpy()
conjunto = catalogo["conjunto"].to_numpy()
ids = catalogo["id"].to_numpy()
print("Dispositivo de treino:", DISPOSITIVO)

## 1. O formato YOLO

Uma caixa (x0, y0, x1, y1) em pixels vira `classe x_centro y_centro largura altura`, tudo dividido pelo
tamanho da imagem. Uma linha por objeto; imagem sem objeto = arquivo vazio. O exercício "de volta para
pixels" do curso é o caminho inverso.

In [ ]:
exemplo = (36.0, 46.0, 98.0, 94.0)
linha = deteccao.para_yolo(exemplo)
print("Caixa em pixels:     ", exemplo)
print("Linha no .txt:        0", " ".join(f"{v:.4f}" for v in linha))
print("De volta para pixels:", tuple(round(v, 1) for v in deteccao.de_yolo(*linha)))

## 2. Pseudo-rótulos: o método clássico rotula, a rede aprende

Para cada raio-X, a caixa do coração vem das bordas que o perfil do ICT mediu (notebook 04): na
horizontal, do percentil 10 da borda direita ao 90 da esquerda; na vertical, da primeira à última linha
medida. Exames sem medida (0,7 %) ficam fora do treino.

Antes de treinar, a pergunta honesta: **quão bons são esses rótulos?** Nos 29 exames com contorno à mão,
compara-se a caixa do pseudo-rótulo com a caixa do contorno.

In [ ]:
medidas = [ict.medir_ict(im, **PARAMS_ICT) for im in imagens]
pseudo = [deteccao.caixa_do_ict(m) for m in medidas]
print(f"Exames com pseudo-rótulo: {sum(p is not None for p in pseudo)} de {len(pseudo)}")

marcacoes = segmentacao.carregar_marcacoes()
gabarito = {r["id"]: deteccao.caixa_do_poligono(r["pontos"]) for _, r in marcacoes.iterrows()}
pos = {i: k for k, i in enumerate(ids)}
qualidade = pd.DataFrame([
    {"id": i, "iou_pseudo_x_mao": deteccao.iou_caixas(pseudo[pos[i]], g) if pseudo[pos[i]] else 0.0,
     "altura_pseudo": (pseudo[pos[i]][3] - pseudo[pos[i]][1]) if pseudo[pos[i]] else np.nan,
     "altura_mao": g[3] - g[1],
     "largura_pseudo": (pseudo[pos[i]][2] - pseudo[pos[i]][0]) if pseudo[pos[i]] else np.nan,
     "largura_mao": g[2] - g[0]}
    for i, g in gabarito.items()])
salvar_tabela(qualidade, SECAO, "qualidade-pseudo-rotulos", indice=False)
qualidade[["iou_pseudo_x_mao", "altura_pseudo", "altura_mao", "largura_pseudo", "largura_mao"]].median().round(2)

In [ ]:
PASTA = config.DATA_PROCESSED / "yolo"
for nome, sel in [("train", "treino"), ("val", "validacao"), ("test", "teste")]:
    for k in np.where(conjunto == sel)[0]:
        if pseudo[k] is not None:
            deteccao.gravar_exemplo(PASTA / nome, ids[k].replace("/", "_"), imagens[k], [pseudo[k]])
for i, g in gabarito.items():
    deteccao.gravar_exemplo(PASTA / "gabarito", i.replace("/", "_"), imagens[pos[i]], [g])

DADOS = deteccao.gravar_yaml(PASTA / "coracao.yaml", PASTA, "train/images", "val/images")
DADOS_TESTE = deteccao.gravar_yaml(PASTA / "teste.yaml", PASTA, "train/images", "test/images")
DADOS_GABARITO = deteccao.gravar_yaml(PASTA / "gabarito.yaml", PASTA, "train/images", "gabarito/images")
print({p.name: len(list((p / "images").glob("*.png"))) for p in sorted(PASTA.iterdir()) if p.is_dir()})

## 3. Treino a partir dos pesos do COCO

YOLOv8n (3 milhões de pesos) pré-treinado no COCO, que não tem a classe "coração". Imagens de 128 px,
até 40 épocas com paciência 10, sem espelhamento nem mosaico; pequenas rotações, deslocamentos e
variação de brilho, como no aumento da CNN.

In [ ]:
PESOS_COCO = config.MODELS / "yolov8n.pt"
if not PESOS_COCO.exists():
    urllib.request.urlretrieve("https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8n.pt", PESOS_COCO)
PROJETO = config.MODELS / "yolo"
MELHOR = PROJETO / "coracao" / "weights" / "best.pt"

if RETREINAR or not MELHOR.exists():
    modelo = YOLO(str(PESOS_COCO))
    modelo.train(data=str(DADOS), epochs=EPOCAS, patience=10, imgsz=config.TAMANHO, batch=64,
                 device=DISPOSITIVO, workers=0, seed=config.SEMENTE, deterministic=True,
                 fliplr=0.0, mosaic=0.0, hsv_h=0.0, hsv_s=0.0, hsv_v=0.2,
                 degrees=5.0, translate=0.05, scale=0.1,
                 project=str(PROJETO), name="coracao", exist_ok=True, plots=False, verbose=False)
detector = YOLO(str(MELHOR))

curvas = pd.read_csv(PROJETO / "coracao" / "results.csv")
curvas.columns = [c.strip() for c in curvas.columns]
melhor_epoca = int(curvas.loc[curvas["val/box_loss"].idxmin(), "epoch"])
salvar_tabela(curvas, SECAO, "curvas-treino", indice=False)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 3.8))
ax1.plot(curvas["epoch"], curvas["train/box_loss"], color=config.AZUL_COBALTO, ls="--", label="treino")
ax1.plot(curvas["epoch"], curvas["val/box_loss"], color=config.AZUL_COBALTO, lw=2, label="validação")
ax1.axvline(melhor_epoca, color=config.CORAL, ls=":", label=f"melhor época ({melhor_epoca})")
ax1.set(xlabel="época", ylabel="box_loss", title="Perda da caixa")
ax1.legend()
ax2.plot(curvas["epoch"], curvas["metrics/mAP50(B)"], color=config.VERDE, lw=2, label="mAP50")
ax2.plot(curvas["epoch"], curvas["metrics/mAP50-95(B)"], color=config.ROXO, lw=2, label="mAP50-95")
ax2.set(xlabel="época", ylabel="mAP (validação, contra pseudo-rótulos)", title="mAP na validação")
ax2.legend()
salvar_figura(fig, SECAO, "curvas-treino")
plt.show()
print(f"{len(curvas)} épocas (parada antecipada); menor val/box_loss na época {melhor_epoca}. "
      "O best.pt é escolhido pelo YOLO pela aptidão (0,1·mAP50 + 0,9·mAP50-95) na validação.")

## 4. Detecção em imagem: confiança e NMS

O detector propõe muitas caixas e dá a cada uma uma **confiança**. Duas regras limpam a lista: o
**limiar de confiança** (descarta as fracas) e o **NMS** (entre caixas que se sobrepõem além de um IoU,
fica só a mais confiante). Para um exame: quantas caixas sobram em cada combinação.

In [ ]:
i_ex = int(np.where(ids == "test/true/339")[0][0])
img_ex = np.stack([imagens[i_ex]] * 3, axis=-1)
combos = [(0.01, 1.0, "conf 0,01 · sem NMS"), (0.01, 0.7, "conf 0,01 · NMS 0,7"),
          (0.25, 0.7, "conf 0,25 · NMS 0,7"), (0.6, 0.7, "conf 0,6 · NMS 0,7")]
fig, eixos = plt.subplots(1, 4, figsize=(15, 4))
contagem = {}
for ax, (conf, iou, titulo) in zip(eixos, combos):
    r = detector.predict(img_ex, imgsz=config.TAMANHO, conf=conf, iou=iou, device=DISPOSITIVO, verbose=False)[0]
    caixas, confs = r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()
    contagem[titulo] = len(caixas)
    ax.imshow(imagens[i_ex], cmap="gray")
    for (x0, y0, x1, y1), cf in zip(caixas, confs):
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec=config.CORAL, lw=1, alpha=0.3 + 0.7 * cf))
    ax.set_title(f"{titulo}\n{len(caixas)} caixas", fontsize=9)
    ax.axis("off")
salvar_figura(fig, SECAO, "confianca-e-nms", pd.DataFrame({"caixas": contagem}))
plt.show()

In [ ]:
def melhor_caixa(indices, conf=0.01):
    """A caixa mais confiante de cada imagem (ou None), com a confiança."""
    lote = [np.stack([imagens[k]] * 3, axis=-1) for k in indices]
    saida = []
    for inicio in range(0, len(lote), 128):
        for r in detector.predict(lote[inicio:inicio + 128], imgsz=config.TAMANHO, conf=conf,
                                  device=DISPOSITIVO, verbose=False):
            if len(r.boxes):
                j = int(r.boxes.conf.argmax())
                saida.append((tuple(float(v) for v in r.boxes.xyxy[j].cpu().numpy()), float(r.boxes.conf[j])))
            else:
                saida.append((None, 0.0))
    return saida

idx_te = np.where(conjunto == "teste")[0]
deteccoes_teste = dict(zip(idx_te, melhor_caixa(idx_te)))
print(f"Exames de teste com alguma caixa: {sum(c is not None for c, _ in deteccoes_teste.values())} de {len(idx_te)}")

fig, eixos = plt.subplots(2, 6, figsize=(15, 5.6))
rng = np.random.default_rng(config.SEMENTE)
for ax, k in zip(eixos.ravel(), rng.choice(idx_te, 12, replace=False)):
    ax.imshow(imagens[k], cmap="gray")
    caixa, cf = deteccoes_teste[k]
    if caixa:
        x0, y0, x1, y1 = caixa
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec=config.CORAL, lw=1.8))
        ax.text(x0, y0 - 2, f"coração {cf:.2f}", color=config.CORAL, fontsize=7)
    ax.set_title(f"{ids[k]} · {config.ROTULOS[y[k]]}", fontsize=7)
    ax.axis("off")
salvar_figura(fig, SECAO, "deteccoes-teste")
plt.show()

## 5. Contra o gabarito: o aluno supera o professor?

Duas réguas nas caixas dos 29 contornos à mão:

* **IoU** da caixa mais confiante de cada exame, lado a lado com o IoU do próprio pseudo-rótulo. Se o
  YOLO só decorasse o método clássico, empataria com ele; se aprendeu o padrão "coração" a partir de
  milhares de exemplos ruidosos, pode passar dele;
* **mAP50** (o `val` do curso): fração de acertos com IoU ≥ 0,5, integrada sobre os limiares de
  confiança. Também contra os pseudo-rótulos do teste, que mede o quanto o aluno reproduz o professor.

In [ ]:
linhas = []
for i, g in gabarito.items():
    k = pos[i]
    caixa_yolo, cf = deteccoes_teste.get(k, (None, 0.0))
    linhas.append({"id": i, "classe": y[k],
                   "iou_pseudo": deteccao.iou_caixas(pseudo[k], g) if pseudo[k] else 0.0,
                   "iou_yolo": deteccao.iou_caixas(caixa_yolo, g) if caixa_yolo else 0.0,
                   "confianca_yolo": cf})
contra_gabarito = pd.DataFrame(linhas)
dif = avaliacao_dif = contra_gabarito["iou_yolo"] - contra_gabarito["iou_pseudo"]
boot = [rng.choice(dif.to_numpy(), len(dif)).mean() for _ in range(2000)]
print(f"IoU médio contra o gabarito: pseudo-rótulo {contra_gabarito['iou_pseudo'].mean():.3f} | "
      f"YOLO {contra_gabarito['iou_yolo'].mean():.3f} | diferença {dif.mean():+.3f} "
      f"(IC 95 % {np.percentile(boot, 2.5):+.3f} a {np.percentile(boot, 97.5):+.3f})")
print(f"Exames com IoU ≥ 0,5: pseudo-rótulo {(contra_gabarito['iou_pseudo'] >= 0.5).mean():.0%} | "
      f"YOLO {(contra_gabarito['iou_yolo'] >= 0.5).mean():.0%}")

metricas = {}
for nome, dados in [("contra o gabarito à mão (29)", DADOS_GABARITO), ("contra os pseudo-rótulos do teste", DADOS_TESTE)]:
    r = detector.val(data=str(dados), imgsz=config.TAMANHO, device=DISPOSITIVO, verbose=False, plots=False,
                     project=str(PROJETO), name="val", exist_ok=True)
    metricas[nome] = {"mAP50": r.box.map50, "mAP50-95": r.box.map, "precisao": r.box.mp, "revocacao": r.box.mr}
metricas = pd.DataFrame(metricas).T
salvar_tabela(contra_gabarito, SECAO, "iou-contra-gabarito", indice=False)
salvar_tabela(metricas, SECAO, "map")
metricas.round(3)

In [ ]:
fig, eixos = plt.subplots(3, 6, figsize=(15, 8.2))
for ax, (_, r) in zip(eixos.ravel(), contra_gabarito.sort_values("iou_yolo").iloc[::2].iterrows()):
    k = pos[r["id"]]
    ax.imshow(imagens[k], cmap="gray")
    for caixa, cor, rot in [(gabarito[r["id"]], config.AMARELO, "à mão"), (pseudo[k], config.AZUL_CLARO, "pseudo"),
                            (deteccoes_teste[k][0], config.CORAL, "YOLO")]:
        if caixa:
            x0, y0, x1, y1 = caixa
            ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec=cor, lw=1.5, label=rot))
    ax.set_title(f"IoU YOLO {r['iou_yolo']:.2f} · pseudo {r['iou_pseudo']:.2f}", fontsize=7)
    ax.axis("off")
for ax in eixos.ravel()[len(contra_gabarito.iloc[::2]):]:
    ax.axis("off")
eixos[0, 0].legend(loc="lower left", fontsize=6)
salvar_figura(fig, SECAO, "caixas-contra-gabarito")
plt.show()

## 6. O ICT da caixa

A largura da caixa do YOLO dividida pela largura do tórax (medida pelo perfil do notebook 04) é um
terceiro jeito de medir o índice cardiotorácico. Nos 1.114 exames de teste, separa as classes melhor
que o ICT do perfil? Comparação pareada nos mesmos exames.

In [ ]:
ict_perfil = np.array([medidas[k].ict for k in idx_te])
ict_caixa = np.array([(deteccoes_teste[k][0][2] - deteccoes_teste[k][0][0]) / medidas[k].torax_px
                      if deteccoes_teste[k][0] and medidas[k].ok else np.nan for k in idx_te])
y_te = y[idx_te]
icts = pd.DataFrame({
    "ICT do perfil (notebook 04)": avaliacao.auc_bootstrap(y_te, ict_perfil),
    "ICT da caixa do YOLO": avaliacao.auc_bootstrap(y_te, ict_caixa),
}).T
par = avaliacao.comparar_auc(y_te, ict_caixa, ict_perfil)
salvar_tabela(icts, SECAO, "ict-da-caixa")
print(f"Caixa − perfil: {par['diferenca']:+.3f} (IC 95 % {par['ic_baixo']:+.3f} a {par['ic_alto']:+.3f}; p = {par['p_valor']:.3f})")
print(f"Mediana do ICT da caixa: normal {np.nanmedian(ict_caixa[y_te == 0]):.2f}, cardiomegalia {np.nanmedian(ict_caixa[y_te == 1]):.2f}")
icts.round(3)

## Conclusões

* **Um dataset próprio sem rotulador humano:** 5.498 dos 5.552 raio-X ganharam caixa pelos
  pseudo-rótulos do ICT. Contra as caixas dos 29 contornos à mão, esses rótulos têm IoU mediano de
  0,48: são do tamanho certo (altura 40 × 41 px, largura 46 × 42 px), mas deslocados e sem o topo do
  coração, porque o perfil só mede as linhas 50 a 90.
* **O YOLO aprende o professor quase perfeitamente:** mAP50 de **0,91** contra os pseudo-rótulos do
  teste. A melhor perda de validação veio na época 18, e a parada antecipada encerrou o treino na 36.
* **Mas não supera o professor:** contra o gabarito à mão, IoU médio **0,497** do YOLO e 0,488 dos
  próprios pseudo-rótulos (diferença +0,008, IC 95 % −0,010 a +0,028); mAP50 de **0,36**. O detector
  herdou o viés do rótulo, inclusive o topo cortado. Em casos isolados ele corrige um pseudo-rótulo
  ruim, mas na média empata. Rótulo ruidoso com o mesmo viés em todos os exemplos não se cancela com
  mais exemplos.
* **Confiança e NMS fazem o trabalho de limpeza:** sem NMS, o mesmo coração recebe 10 caixas
  sobrepostas; com NMS 0,7 sobra uma, e o limiar de confiança de 0,25 a 0,6 não muda essa caixa.
* **O ICT da caixa separa pior:** AUC **0,702** contra 0,739 do ICT do perfil (diferença −0,037,
  IC −0,065 a −0,009; p = 0,009). A caixa é a moldura: arredonda a largura do coração para o retângulo
  e perde o detalhe que a borda linha a linha preserva.
* **Parte do erro é do gabarito:** os piores IoU são os contornos 1 a 4, pequenos demais (notebook 08).
  Com contornos refeitos e mais anotadores, o mAP50 contra o gabarito subiria, sem mexer no detector.

**Lição do curso:** treinar o YOLO com dataset próprio é fácil; o difícil é o dataset. Com
pseudo-rótulos, o detector é tão bom quanto quem rotulou. Para passar do método clássico seria preciso
rotular à mão uma parte do treino, nem que fossem algumas centenas de caixas.